## Word2Vec Implementation

In [1]:
# importing libraries
import numpy as np
from collections import Counter

In [2]:
# small corpus
corpus = [
    "the king ruled the kingdom",
    "the queen ruled the kingdom", 
    "the king sat on the throne",
    "the queen sat on the throne",
    "paris is the capital of france",
    "london is the capital of england",
    "the king of france visited paris",
    "the queen of england visited london",
]

# building the vocabulary
words = " ".join(corpus).split()
vocab = list(set(words))
vocab_size = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {vocab}")

Vocabulary size: 16
Vocabulary: ['sat', 'kingdom', 'queen', 'england', 'ruled', 'capital', 'paris', 'throne', 'london', 'the', 'king', 'of', 'france', 'visited', 'on', 'is']


In [3]:
# generating skip-gram training pairs
def generate_training_pairs(corpus, word_to_idx, window_size=2):
    pairs = []
    for sentence in corpus:
        words = sentence.split()
        for i, center_word in enumerate(words):
            for j in range(max(0, i-window_size), 
                          min(len(words), i+window_size+1)):
                if i != j:
                    context_word = words[j]
                    pairs.append((word_to_idx[center_word], 
                                 word_to_idx[context_word]))
    return pairs

pairs = generate_training_pairs(corpus, word_to_idx, window_size=2)
print(f"Total training pairs: {len(pairs)}")
print(f"First 5 pairs:")
for center, context in pairs[:5]:
    print(f"  '{idx_to_word[center]}' → '{idx_to_word[context]}'")

Total training pairs: 136
First 5 pairs:
  'the' → 'king'
  'the' → 'ruled'
  'king' → 'the'
  'king' → 'ruled'
  'king' → 'the'


In [4]:
# model
class Word2Vec:
    def __init__(self, vocab_size, embedding_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        self.W1 = np.random.randn(vocab_size, embedding_dim) * 0.01  # embedding layer
        self.W2 = np.random.randn(embedding_dim, vocab_size) * 0.01  # prediction layer

    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

    def forward(self, center_idx):
    
        self.h = self.W1[center_idx]
        self.u = self.h @ self.W2
        self.y_pred = self.softmax(self.u)
        return self.y_pred

    def backward(self, center_idx, context_idx, learning_rate=0.01):
        # error
        e = self.y_pred.copy()
        e[context_idx] -= 1                   # subtracting 1 at true context position

        # gradients
        dW2 = np.outer(self.h, e)             # (embedding_dim, vocab_size)
        dW1 = self.W2 @ e                     # (embedding_dim,)

        # update
        self.W2 -= learning_rate * dW2
        self.W1[center_idx] -= learning_rate * dW1  # only update the center word row

    def train(self, pairs, epochs=1000, learning_rate=0.01):
        losses = []
        for epoch in range(epochs):
            total_loss = 0
            for center_idx, context_idx in pairs:
                y_pred = self.forward(center_idx)
                loss = -np.log(y_pred[context_idx] + 1e-8)
                total_loss += loss
                self.backward(center_idx, context_idx, learning_rate)
            if epoch % 100 == 0:
                losses.append(total_loss/len(pairs))
                print(f"Epoch {epoch}: loss={total_loss/len(pairs):.4f}")
        return losses

    def get_embedding(self, word):
        return self.W1[word_to_idx[word]]

    def most_similar(self, word, top_n=5):
        vec = self.get_embedding(word)
        similarities = {}
        for w, idx in word_to_idx.items():
            if w != word:
                other_vec = self.W1[idx]
                cos_sim = np.dot(vec, other_vec) / (
                    np.linalg.norm(vec) * np.linalg.norm(other_vec) + 1e-8)
                similarities[w] = cos_sim
        return sorted(similarities.items(), 
                      key=lambda x: x[1], reverse=True)[:top_n]

# training
model = Word2Vec(vocab_size=vocab_size, embedding_dim=10)
losses = model.train(pairs, epochs=1000, learning_rate=0.05)

# testing similarity
print("\nMost similar to 'king':")
for word, sim in model.most_similar('king'):
    print(f"  {word}: {sim:.4f}")

print("\nMost similar to 'paris':")
for word, sim in model.most_similar('paris'):
    print(f"  {word}: {sim:.4f}")

Epoch 0: loss=2.7726
Epoch 100: loss=1.9392
Epoch 200: loss=1.9436
Epoch 300: loss=1.9442
Epoch 400: loss=1.9458
Epoch 500: loss=1.9477
Epoch 600: loss=1.9491
Epoch 700: loss=1.9498
Epoch 800: loss=1.9500
Epoch 900: loss=1.9496

Most similar to 'king':
  queen: 0.6308
  kingdom: 0.5018
  capital: 0.4661
  paris: 0.3143
  throne: 0.2898

Most similar to 'paris':
  london: 0.6533
  of: 0.3801
  kingdom: 0.3438
  capital: 0.3185
  king: 0.3143


In [5]:
# vector arithmetic
def analogy(word_a, word_b, word_c, top_n=3):

    vec = (model.get_embedding(word_a) 
           - model.get_embedding(word_b) 
           + model.get_embedding(word_c))
    
    similarities = {}
    for w, idx in word_to_idx.items():
        if w not in [word_a, word_b, word_c]:
            other_vec = model.W1[idx]
            cos_sim = np.dot(vec, other_vec) / (
                np.linalg.norm(vec) * np.linalg.norm(other_vec) + 1e-8)
            similarities[w] = cos_sim
    
    return sorted(similarities.items(), 
                  key=lambda x: x[1], reverse=True)[:top_n]

print("king - man + woman = ?")

print("king - ruled + sat = ?")
for word, sim in analogy('king', 'ruled', 'sat'):
    print(f"  {word}: {sim:.4f}")

print("\nparis - france + england = ?")
for word, sim in analogy('paris', 'france', 'england'):
    print(f"  {word}: {sim:.4f}")

king - man + woman = ?
king - ruled + sat = ?
  queen: 0.7175
  throne: 0.4606
  kingdom: 0.2431

paris - france + england = ?
  london: 0.3875
  capital: 0.2304
  of: 0.2188
